# 03 — Model Training

Two-stage transfer learning pipeline using **EfficientNetV2-S**:

**Stage 1 — EyePACS pre-training** (3-phase progressive unfreezing):
- **Phase 1**: Head-only with light augmentation (frozen backbone)
- **Phase 2**: Last block unfrozen (`blocks[-1:]`) with full augmentation + Mixup
- **Phase 3**: Full unfreeze with full augmentation + Mixup

Key features:
- Patient-level stratified splitting (no data leakage)
- Combined loss: Ordinal cross-entropy + Focal loss
- Exponential Moving Average (EMA)
- Cosine LR with warmup + discriminative LR
- Checkpointing on Quadratic Weighted Kappa (QWK)


In [ ]:
import importlib
import os
import sys

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle scipy
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config
config.ensure_dirs()


In [ ]:
import torch
import pandas as pd

from src.augmentations import get_light_train_transforms, get_train_transforms, get_valid_transforms
from src.dataset import (
    compute_class_weights,
    create_dataloaders,
    load_and_split_dataframe,
    preprocess_images_to_cache,
)
from src.model import (
    ModelEMA,
    build_model,
    count_parameters,
    freeze_backbone,
    unfreeze_backbone,
)
from src.trainer import Trainer
from src.utils import get_device, log_gpu_memory, seed_everything

seed_everything()
device = get_device()
log_gpu_memory("Initial")


## 1. Data Preparation


In [ ]:
# Load and split EyePACS dataset (patient-level stratification)
train_df, valid_df, test_df = load_and_split_dataframe(dataset_name="eyepacs")

print(f'Train : {len(train_df):,} images  ({train_df["patient_id"].nunique():,} patients)')
print(f'Valid : {len(valid_df):,} images  ({valid_df["patient_id"].nunique():,} patients)')
print(f'Test  : {len(test_df):,} images  ({test_df["patient_id"].nunique():,} patients)')
print('\nTrain class distribution:')
print(train_df["level"].value_counts().sort_index())


## 2. Preprocessing Cache + DataLoaders

Images are preprocessed once (crop + CLAHE + Ben Graham at 384x384) and cached to disk.
`DRDataset` auto-detects the cache and reads directly from it (~6x speedup).

Two sets of loaders:
- `loaders_light`: light augmentation for Phase 1 (head-only)
- `loaders_heavy`: full augmentation for Phase 2/3 (fine-tuning)


In [ ]:
# Build preprocessing cache (runs once, skips existing images)
print("Building/verifying preprocessed cache...")
preprocess_images_to_cache(train_df, dataset_name="eyepacs")
preprocess_images_to_cache(valid_df, dataset_name="eyepacs")
print("Cache ready.")


In [ ]:
# Transforms
light_tfm = get_light_train_transforms(config.IMG_SIZE)
heavy_tfm = get_train_transforms(config.IMG_SIZE)
valid_tfm = get_valid_transforms(config.IMG_SIZE)

# Phase 1 loaders: light augmentation
loaders_light = create_dataloaders(
    train_df=train_df,
    valid_df=valid_df,
    dataset_name="eyepacs",
    train_transform=light_tfm,
    valid_transform=valid_tfm,
    batch_size=config.STAGE1_BATCH_SIZE,
)

# Phase 2/3 loaders: full augmentation
loaders_heavy = create_dataloaders(
    train_df=train_df,
    valid_df=valid_df,
    dataset_name="eyepacs",
    train_transform=heavy_tfm,
    valid_transform=valid_tfm,
    batch_size=config.STAGE1_BATCH_SIZE,
)

print(f"Train batches (light): {len(loaders_light['train'])}")
print(f"Train batches (heavy): {len(loaders_heavy['train'])}")
print(f"Valid batches:         {len(loaders_light['valid'])}")


## 3. Model + Class Weights


In [ ]:
model = build_model(pretrained=True)
params = count_parameters(model)

print(f"Model: {config.MODEL_NAME}")
print(f"Input size: {config.IMG_SIZE}x{config.IMG_SIZE}")
print(f"Total parameters: {params['total']:,}")
print(f"Stochastic depth: {config.STOCHASTIC_DEPTH_RATE}")
print(f"Dropout rate: {config.DROPOUT_RATE}")
print(f"\nLoss: Combined (OrdinalCE + FocalLoss)")
print(f"  Focal gamma: {config.FOCAL_GAMMA}")
print(f"  Label smoothing: {config.LABEL_SMOOTHING}")
print(f"  Class-weighted: {config.USE_CLASS_WEIGHTED_LOSS}")
print(f"\nMixup: {config.USE_MIXUP} (alpha={config.MIXUP_ALPHA})")
print(f"  Class-conditional: {config.USE_CLASS_CONDITIONAL_MIXUP}")
print(f"\nEMA: {config.USE_EMA} (decay={config.EMA_DECAY})")


In [ ]:
# Class weights for loss function
class_weights = compute_class_weights(train_df["level"].values)
print("Class weights:")
for name, w in zip(config.CLASS_NAMES, class_weights):
    print(f"  {name:20s}: {w:.4f}")


## 4. Stage 1 — EyePACS Training

| Phase | Backbone | Augmentation | LR | WD | Epochs | Mixup |
|-------|----------|-------------|-----|-----|--------|-------|
| 1 | Frozen (head-only) | Light | 1e-3 | 1e-4 | 8 | No |
| 2 | blocks[-1:] (72%) | Full | 5e-5 | 1e-4 | 12 | Yes |
| 3 | Full unfreeze | Full | 1e-5 | 5e-4 | 15 | Yes |

Phase 2/3: head gets 10x the backbone LR (discriminative learning rates).


In [ ]:
# Create EMA and Trainer
ema = ModelEMA(model) if config.USE_EMA else None

trainer = Trainer(
    model=model,
    device=device,
    class_weights=class_weights,
    ema=ema,
)

log_gpu_memory("After model init")


### Phase 1: Head-Only Training


In [ ]:
freeze_backbone(model)
print(f"Phase 1 trainable: {count_parameters(model)}")

trainer.fit_phase(
    phase="s1_phase1",
    train_loader=loaders_light["train"],
    valid_loader=loaders_light["valid"],
    epochs=config.STAGE1_PHASE1_EPOCHS,
    lr=config.STAGE1_PHASE1_LR,
    weight_decay=config.STAGE1_PHASE1_WD,
    use_mixup=False,
)


### Phase 2: Partial Unfreeze (Last Block)


In [ ]:
trainer.load_best_checkpoint(phase="s1_phase1")
unfreeze_backbone(model, from_block=config.STAGE1_PHASE2_UNFREEZE)
print(f"Phase 2 trainable: {count_parameters(model)}")

trainer.fit_phase(
    phase="s1_phase2",
    train_loader=loaders_heavy["train"],
    valid_loader=loaders_heavy["valid"],
    epochs=config.STAGE1_PHASE2_EPOCHS,
    lr=config.STAGE1_PHASE2_LR,
    weight_decay=config.STAGE1_PHASE2_WD,
    warmup_epochs=2,
    use_mixup=config.USE_MIXUP,
)


### Phase 3: Full Unfreeze


In [ ]:
trainer.load_best_checkpoint(phase="s1_phase2")
unfreeze_backbone(model, from_block=None)
print(f"Phase 3 trainable: {count_parameters(model)}")

trainer.fit_phase(
    phase="s1_phase3",
    train_loader=loaders_heavy["train"],
    valid_loader=loaders_heavy["valid"],
    epochs=config.STAGE1_PHASE3_EPOCHS,
    lr=config.STAGE1_PHASE3_LR,
    weight_decay=config.STAGE1_PHASE3_WD,
    warmup_epochs=2,
    use_mixup=config.USE_MIXUP,
)


## 5. Training Curves


In [ ]:
from src.evaluate import plot_training_curves

plot_training_curves(
    trainer.history,
    save_path=os.path.join(config.FIGURES_DIR, "stage1_training_curves.png"),
    phase_boundaries=[
        config.STAGE1_PHASE1_EPOCHS,
        config.STAGE1_PHASE1_EPOCHS + config.STAGE1_PHASE2_EPOCHS,
    ],
)

trainer.save_history(os.path.join(config.LOGS_DIR, "stage1_history.json"))


## 6. Quick Validation


In [ ]:
from src.evaluate import compute_metrics, predict

trainer.load_best_checkpoint("s1_phase3")

if ema is not None:
    ema.apply(model)
    print("Applied EMA weights")

y_true, y_pred, y_proba = predict(model, loaders_heavy["valid"], device)
metrics = compute_metrics(y_true, y_pred, y_proba)

print(f"Validation QWK      : {metrics['qwk']:.4f}")
print(f"Validation Accuracy : {metrics['accuracy']:.4f}")
print(f"Validation ROC-AUC  : {metrics['roc_auc_weighted']:.4f}")
print(f"\n{metrics['classification_report_str']}")


## 7. Save Test Split


In [ ]:
test_csv_path = os.path.join(config.OUTPUTS_DIR, "test_split_eyepacs.csv")
test_df.to_csv(test_csv_path, index=False)
print(f"Test split saved: {test_csv_path} ({len(test_df)} images)")


## 8. ONNX Export

Export the best model to ONNX for production deployment.


In [ ]:
from src.export import export_to_onnx

onnx_path = export_to_onnx(
    checkpoint_path=os.path.join(config.CHECKPOINTS_DIR, "best_model_s1_phase3.pth"),
)
print(f"\nONNX model exported to: {onnx_path}")


## 9. Next Steps

Stage 1 (EyePACS pre-training) complete. Continue with:
- `python train_pipeline.py --stage2-only` for Stage 2 (APTOS fine-tuning)
- `04_evaluation.ipynb` for full evaluation
- `05_gradcam.ipynb` for explainability
